## Module 4-1 NLP Basics and Regular Expressions

In [ ]:
import pandas as pd
from pathlib import Path
from nltk.tokenize import sent_tokenize, word_tokenize

*Run the following codes for the first time you use the `nltk` package*

nltk.download('punkt')

nltk.download('cmudict')

nltk.download('punkt_tab')

### 1. Intro to Regex

Regular expressions (regex) are patterns used to match, search, extract, or replace text.

In Python, regex functionality is provided by the `re` module.

In [ ]:
import re

#### 1.1. Basic Regex Concepts

Some of the commonly used patterns are as follows:

| Regex Symbol | Meaning                                      | Example                                    | 
| ------------ | -------------------------------------------- | ------------------------------------------ |
| `.`          | Any character except newline                 | `a.b` matches "acb", "a1b"                 | 
| `\d`         | Digit (0–9)                                  | `\d{4}` matches "2023"                     | 
| `\w`         | Word character (letters, digits, underscore) | `\w+` matches "Earnings2023"               |
| `\s`         | Whitespace (space, tab, newline)             | `\s+` matches spaces                       | 
| `[]`         | Character class (any inside)                 | `[A-Z]` matches capital letters            | 
| `^`          | Start of string/line                         | `^Item` matches lines starting with "Item" | 
| `$`          | End of string/line                           | `profit$` matches lines ending in "profit" | 
| `*`          | 0 or more repetitions                        | `ab*` matches "a", "ab", "abb"             | 
| `+`          | 1 or more repetitions                        | `\d+` matches "2023", "100"                | 
| `{m,n}`      | Between m and n repetitions                  | `\d{2,4}` matches "23", "2023"             | 
| `()`         | Group (capture)                              | `(20\d{2})` captures years like 2019       | 
| `\|`         | OR                                           | \`(profit \| loss)\` matches either word   |

You can refer to the Regular Expressions Cheat Sheet at https://cheatography.com/davechild/cheat-sheets/regular-expressions/.

#### 1.2. Identifying Patterns in Texts using `re.findall()`

Basic syntax
```py
list_of_str_found = re.findall(r"pattern", existing_str_var)
```

In [ ]:
text = '''
The company reported earnings of $5.2 billion [15] and revenues 
of $20 billion in 2023 [16].
'''

In [ ]:
# Extract monetary amounts as full strings
amounts = re.findall(r"\$\d+(?:\.\d+)?\s*(?:thousand|million|billion)", text)
print(amounts)

In [ ]:
# Find financial terms like "earnings", "revenue", "loss"
# Use `flag=re.IGNORECASE` to make the search case-insensitive
re.findall(r"\b(earnings?|revenues?|loss(?:es)?)\b", text, flags=re.IGNORECASE)

In [ ]:
# Find footnotes look like [1], (2), or superscripts ^2.
re.findall(r"(\[\d+\]|\(\d+\)|\^\d+)", text)

In [ ]:
# Find forward-looking statements with keywords such as "expect", "anticipate", "may"
text = "We expect revenues to increase in the future, but results may differ."

forward_looking = r"\b(expects?|anticipates?|believes?|future|may|might|could|intend|plan|predict)\b"
re.findall(forward_looking, text, flags=re.IGNORECASE)

In [ ]:
if len(re.findall(forward_looking, text, flags=re.IGNORECASE)) > 0:
    print(text)

In [ ]:
# Similar to, but more advanced than:
Future_terms = [
    "expect", "anticipate", "believe", "future", "may",
    "might", "could", "intend", "plan", "predict"
    ]

[term for term in Future_terms if term in text.lower()]

#### 1.3. Cleaning Texts using `re.sub()`

Basic syntax
```py
new_str_var = re.sub(r"text to be replaced", "new content", old_str_var)
```

In [ ]:
# Remove tables (such as lines full of dashes, pipes, or repeated symbols)

raw_text = """
-------------------------------------------------
| Quarter | Revenue ($m) | Profit ($m) |
-------------------------------------------------
Q1        1000           200
Forward-looking statements appear below.

Item 1: Business Overview
"""

In [ ]:
cleaned1 = re.sub(r"[-|=_]{3,}", "", raw_text)
print(cleaned1)

In [ ]:
# Remove formatting artifacts (extra spaces, newlines)
cleaned2 = re.sub(r"(\||\s+)", " ", cleaned1)
cleaned2 = cleaned2.strip()
print(cleaned2)

In [ ]:
# Remove boilerplate disclaimers
# Most 10-Ks include a “Safe Harbor” or “Forward-Looking Statements” section.

pattern = r"forward-looking statements.*?(?=item\s1)"  # until "Item 1"
cleaned3 = re.sub(pattern, "", cleaned2, flags=re.IGNORECASE | re.DOTALL)
print(cleaned3)

In [ ]:
# Example Project: Cleaning an MD&A section

mda = """
Item 7. Management Discussion & Analysis
----------------------------------------
Our revenues increased by 10% in 2022 [1].
We expect growth in the future, but results may differ materially.

-------------------------------------------------
| Quarter | Revenue ($m) | Profit ($m) |
-------------------------------------------------
Q1        1000           200
Q2        1100           250
Q3        1200           300
Q4        1300           350
----------------------------------------
Safe Harbor Statement: These forward-looking statements are subject to risks...

Item 7A. Quantitative and Qualitative Disclosures
"""

In [ ]:
# Remove tables
mda = re.sub(r"[-|=_]{3,}.*", "", mda)

# Remove footnotes
mda = re.sub(r"(\[\d+\]|\(\d+\))", "", mda)

# Remove boilerplate Safe Harbor
mda = re.sub(r"Safe Harbor.*?(?=Item 7A)", "", mda, flags=re.DOTALL | re.IGNORECASE)

print(mda)

### 2. Introduction to `textstat`

See the offical documents at https://github.com/textstat/textstat.

In [ ]:
import textstat

In [ ]:
# Sample texts

text_easy = (
    "Robots help in the warehouse. They lift boxes and move them to the right shelves. "
    "Workers use tablets to check orders. The system is simple to learn and fast to use."
)

text_medium = (
    "Management expects revenue to grow modestly as the company reallocates resources toward higher-margin segments. "
    "While this transition may reduce short-term sales, it is designed to enhance operating leverage and cash flows."
)

text_hard = (
    "The amortization of capitalized development expenditures, coupled with deferred tax adjustments arising from "
    "temporary timing differences, materially affected the period's comprehensive income and diluted earnings trajectory."
)

#### 2.1. Basic counts and estimates

`textstat` offers quick utilities for character, word, syllable, and sentence counts, plus estimated reading time.

In [ ]:
def basic_metrics(text: str) -> dict:
    return {
        "chars": textstat.char_count(text, ignore_spaces=True),
        "letters": textstat.letter_count(text),
        "words": textstat.lexicon_count(text, removepunct=True),
        "sentences": textstat.sentence_count(text),
        "syllables": textstat.syllable_count(text),
        "avg_sentence_length": textstat.words_per_sentence(text),
        "avg_syllables_per_word": textstat.avg_syllables_per_word(text),
    }

basic_metrics(text_medium)

#### 2.2. Calculating Readability scores (single text)

Here are the most commonly cited readability indices. Lower complexity → higher *Flesch Reading Ease*; higher grades → more difficult.

- **Gunning Fog** (`gunning_fog`)
- **SMOG Index** (`smog_index`)
- **Flesch Reading Ease** (`flesch_reading_ease`) — higher is easier (typical range ≈ 0–100).
- **Flesch–Kincaid Grade** (`flesch_kincaid_grade`)
- **Coleman–Liau** (`coleman_liau_index`)
- **Automated Readability Index** (`automated_readability_index`)
- **Dale–Chall Score** (`dale_chall_readability_score`)
- **Linsear Write** (`linsear_write_formula`)
- **Text Standard** (`text_standard`) — `textstat`’s combined “grade band” summary (string).

In [ ]:
def readability_metrics(text: str) -> dict:
    return {
        "gunning_fog": textstat.gunning_fog(text),
        "smog_index": textstat.smog_index(text),
        "flesch_reading_ease": textstat.flesch_reading_ease(text),
        "flesch_kincaid_grade": textstat.flesch_kincaid_grade(text),
        "coleman_liau": textstat.coleman_liau_index(text),
        "automated_readability": textstat.automated_readability_index(text),
        "dale_chall": textstat.dale_chall_readability_score(text),
        "linsear_write": textstat.linsear_write_formula(text),
        "text_standard": textstat.text_standard(text),
    }

easy = readability_metrics(text_easy)
medium = readability_metrics(text_medium)
hard = readability_metrics(text_hard)

pd.DataFrame([easy, medium, hard], index=["Easy", "Medium", "Hard"])

#### 2.3. Difficult words and vocabulary insights

- `difficult_words(text)` returns a count based on the New Dale–Chall list.
- `polysyllabcount(text)` counts words with ≥3 syllables.
- `monosyllabcount(text)` counts one-syllable words.

In [ ]:
def vocab_metrics(text: str) -> dict:
    return {
        "difficult_word_count": textstat.difficult_words(text),
        "polysyllables": textstat.polysyllabcount(text),
        "monosyllables": textstat.monosyllabcount(text),
        "lexicon_count": textstat.lexicon_count(text, removepunct=True),
        "syllables": textstat.syllable_count(text),
    }

vocab_metrics(text_hard)

### 3. Other useful packages to learn:

- `spacy`: https://spacy.io/usage/spacy-101
- `nltk`: https://www.nltk.org/